In [ ]:
import os, io, re, json, base64, datetime as dt
import requests
import pandas as pd
import matplotlib.pyplot as plt
import asyncio, websockets

# -------- Config --------
NIM_BASE  = os.getenv("NIM_BASE", "http://localhost:8000").rstrip("/")
NIM_KEY   = os.getenv("NIM_KEY",  "nim-local-key")
NIM_MODEL = os.getenv("NIM_MODEL","weathermanj/nemotron-nano-9b-fp8")

MCP_URL       = os.getenv("MCP_URL", "ws://localhost:5173")
MCP_SQL_TOOL  = os.getenv("MCP_SQL_TOOL", "mssql.run_sql")

# Task + schema tuned to dbo.Call_Data
TASK = os.getenv("TASK", "Show weekly call counts grouped by ingress_devs_names for the last 12 weeks.")
SCHEMA_HINT = os.getenv("SCHEMA_HINT",
    "dbo.Call_Data("
    "date DATETIME, "
    "ingress_devs_names VARCHAR, "
    "src_user VARCHAR, "
    "callee_ip_name VARCHAR, "
    "call_time FLOAT"
    "); Each row is one call; date is the call timestamp; call_time is duration (seconds)."
)

# -------- HTTP util --------
def _post_json(url: str, payload: dict, timeout=120) -> requests.Response:
    headers = {"Authorization": f"Bearer {NIM_KEY}", "Content-Type": "application/json"}
    return requests.post(url, headers=headers, json=payload, timeout=timeout)

# -------- LLM helpers --------
def llm_chat_or_completion(messages, temperature=0.2, max_tokens=512) -> str:
    # Try chat first
    try:
        r = _post_json(f"{NIM_BASE}/v1/chat/completions",
                        {"model": NIM_MODEL, "messages": messages,
                        "temperature": temperature, "max_tokens": max_tokens})
        if r.status_code == 200:
            j = r.json()
            return j.get("choices",[{}])[0].get("message",{}).get("content","").strip()
    except Exception as e:
        print(f"[warn] chat call failed: {e}")

    # Fallback to completions
    try:
        sys = "\n".join(m["content"] for m in messages if m["role"] == "system")
        usr = "\n\n".join(m["content"] for m in messages if m["role"] == "user")
        prompt = (f"System:\n{sys}\n\nUser:\n{usr}") if sys else usr
        r2 = _post_json(f"{NIM_BASE}/v1/completions",
                            {"model": NIM_MODEL, "prompt": prompt,
                            "temperature": temperature, "max_tokens": max_tokens})
        if r2.status_code == 200:
            return r2.json().get("choices",[{}])[0].get("text","").strip()
        else:
            print(f"[warn] completions HTTP {r2.status_code}: {r2.text[:300]}")
    except Exception as e:
        print(f"[warn] completions call failed: {e}")
    return ""

# -------- SQL extraction & fallback builder --------
_CODE_FENCE = re.compile(r"^```(?:sql)?\s*|\s*```$", flags=re.IGNORECASE|re.MULTILINE)
_SELECT_RE  = re.compile(r"(?is)\bselect\b.+?(?:;|$)")

def extract_select_sql(text: str) -> str | None:
    if not text: return None
    cleaned = _CODE_FENCE.sub("", text).strip()
    m = _SELECT_RE.search(cleaned)
    if not m: return None
    sql = " ".join(m.group(0).strip().split())
    if sql.endswith(";"): sql = sql[:-1]
    return sql if sql[:6].lower()=="select" else None

def parse_schema_hint(schema_hint: str):
    """Very light parser: return (table, [cols])."""
    m = re.search(r'([A-Za-z0-9_.]+\.)?[A-Za-z0-9_]+\s*\(([^)]+)\)', schema_hint)
    if not m:
        return None, []
    table = schema_hint[m.start(): m.end()].split("(")[0].strip()
    cols_block = m.group(2)
    cols = []
    for raw in cols_block.split(","):
        name = raw.strip().split()[0].strip("[]")
        if name: cols.append(name)
    return table, cols

def build_fallback_sql(task: str, schema_hint: str) -> str:
    """Weekly counts, grouped by ingress_devs_names if present."""
    table, cols = parse_schema_hint(schema_hint)
    if not table:
        return "SELECT TOP (100) * FROM sys.tables"

    lower = [c.lower() for c in cols]
    date_col = next((c for c in cols if any(k in c.lower() for k in ("created_at","timestamp","date","time"))), None)
    ingress_col = "ingress_devs_names" if "ingress_devs_names" in lower else None
    status_col = next((c for c in cols if c.lower()=="status"), None)

    if date_col:
        where = f"WHERE {date_col} >= DATEADD(week,-12,SYSUTCDATETIME())"
        if "active" in task.lower() and status_col:
            where += f" AND {status_col} = 'active'"

        # group by ingress if available
        select_prefix = f"{ingress_col}, " if ingress_col else ""
        group_suffix  = f", {ingress_col}" if ingress_col else ""

        sql = f"""
        SELECT
            {select_prefix}DATEADD(week, DATEDIFF(week, 0, {date_col}), 0) AS week_start,
            COUNT(*) AS value
        FROM {table}
        {where}
        GROUP BY DATEADD(week, DATEDIFF(week, 0, {date_col}), 0){group_suffix}
        ORDER BY week_start{group_suffix}
        """
        return " ".join(sql.split())

    # No date column? return a simple top snapshot
    first_two = ", ".join(cols[:2]) if len(cols) >= 2 else cols[0]
    return f"SELECT TOP (100) {first_two} FROM {table}"

def get_select_sql(task: str, schema_hint: str) -> str:
    system = {"role":"system","content":(
        "Output exactly ONE Microsoft SQL Server SELECT statement on a single line. "
        "No commentary/markdown. Use ISO dates. No DDL/DML."
    )}
    user = {"role":"user","content":f"Task: {task}\nSchema: {schema_hint}\nOne SELECT only."}
    text = llm_chat_or_completion([system, user]).strip()
    sql = extract_select_sql(text)
    if sql:
        return sql

    system2 = {"role":"system","content":"Return ONLY one line of SQL that starts with SELECT. Nothing else."}
    user2 = {"role":"user","content":f"Task: {task}\nSchema: {schema_hint}\nONE LINE SELECT ONLY."}
    text2 = llm_chat_or_completion([system2, user2], temperature=0.0, max_tokens=400).strip()
    sql2 = extract_select_sql(text2)
    if sql2:
        return sql2

    print("[warn] Model returned no usable SELECT; using heuristic fallback.")
    fb = build_fallback_sql(task, schema_hint)
    print(f"[info] Fallback SQL: {fb}")
    return fb

# -------- MCP helpers --------
async def mcp_call(tool_name: str, args: dict):
    async with websockets.connect(MCP_URL, ping_interval=20) as ws:
        await ws.send(json.dumps({"jsonrpc":"2.0","id":1,"method":"tools/list"}))
        _ = await ws.recv()
        await ws.send(json.dumps({
            "jsonrpc":"2.0","id":2,"method":"tools/call",
            "params":{"name": tool_name, "arguments": args}
        }))
        raw = await ws.recv()
        msg = json.loads(raw)
        if "error" in msg:
            raise RuntimeError(msg["error"])
        return msg["result"]

def run_sql(sql: str) -> pd.DataFrame:
    res = asyncio.get_event_loop().run_until_complete(
        mcp_call(MCP_SQL_TOOL, {"sql": sql})
    )
    if "columns" in res and "rows" in res:
        cols, rows = res["columns"], res["rows"]
    elif "schema" in res and "rows" in res:
        cols = [c["name"] for c in res["schema"]]; rows = res["rows"]
    else:
        raise ValueError(f"Unexpected MCP result shape: {res.keys()}")
    return pd.DataFrame(rows, columns=cols)

# -------- plotting --------
def fig_to_base64(fig) -> str:
    buf = io.BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig)
    buf.seek(0); return base64.b64encode(buf.read()).decode("ascii")

def simple_chart(df: pd.DataFrame) -> tuple[str, str]:
    cols = [c.lower() for c in df.columns]
    # detect time + group columns
    date_col = next((c for c in df.columns if any(k in c.lower() for k in ("week_start","date","timestamp","time"))), None)
    group_col = None
    for cand in ("ingress_devs_names","src_user","callee_ip_name"):
        if cand in cols:
            group_col = df.columns[cols.index(cand)]
            break

    # numeric measure
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    value_col = None
    for cand in ("value","count","calls","call_count","duration","call_time"):
        if cand in [c.lower() for c in df.columns]:
            value_col = df.columns[[c.lower() for c in df.columns].index(cand)]
            break
    if not value_col and numeric_cols:
        value_col = numeric_cols[0]

    if date_col and value_col:
        if group_col:
            # multi-series line: pivot wide by group
            pivot = (
                df
                .assign(**{date_col: pd.to_datetime(df[date_col], errors="coerce")})
                .pivot_table(index=date_col, columns=group_col, values=value_col, aggfunc="sum")
                .sort_index()
            )
            fig = plt.figure(); ax = fig.add_subplot(111)
            pivot.plot(ax=ax)
            ax.set_title(f"{value_col} over time by {group_col}")
            ax.set_xlabel(date_col); ax.set_ylabel(value_col); ax.grid(True, alpha=0.3)
            return fig_to_base64(fig), f"Time series of {value_col} grouped by {group_col}."
        else:
            # single-series line
            s = (
                df
                .assign(**{date_col: pd.to_datetime(df[date_col], errors="coerce"),
                            value_col: pd.to_numeric(df[value_col], errors="coerce")})
                .sort_values(date_col)
            )
            fig = plt.figure(); ax = fig.add_subplot(111)
            ax.plot(s[date_col], s[value_col])
            ax.set_title(f"{value_col} over {date_col}")
            ax.set_xlabel(date_col); ax.set_ylabel(value_col); ax.grid(True, alpha=0.3)
            return fig_to_base64(fig), f"Time series of {value_col}."

    # fallback: top-20 bar of first two columns
    x = df.columns[0]
    y = df.columns[1] if len(df.columns) > 1 else df.columns[0]
    fig = plt.figure(); ax = fig.add_subplot(111)
    ax.bar(df[x].astype(str).head(20), pd.to_numeric(df[y], errors="coerce").head(20))
    ax.set_title(f"{y} by {x} (top 20)")
    ax.set_xlabel(x); ax.set_ylabel(y); ax.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=30, ha="right")
    return fig_to_base64(fig), f"Bar chart of {y} by {x} (top 20)."

# -------- main --------
def main():
    print(f"[info] LLM: {NIM_MODEL} @ {NIM_BASE}")
    print(f"[info] MCP: {MCP_URL} tool={MCP_SQL_TOOL}")
    print(f"[info] TASK: {TASK}")
    print(f"[info] HINT: {SCHEMA_HINT}")

    sql = get_select_sql(TASK, SCHEMA_HINT)
    print(f"[info] SQL => {sql}")

    df = run_sql(sql)
    if df.empty:
        raise RuntimeError("Query returned no rows. Tweak TASK/SCHEMA_HINT or widen the window.")
    print(f"[info] Rows={len(df)} Cols={list(df.columns)}")

    img_b64, caption = simple_chart(df)

    now = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
    html = f"""<!doctype html><html><head><meta charset="utf-8"><title>Auto Report</title>
<style>body{{font-family:system-ui,Segoe UI,Roboto,Arial;margin:24px}}h1{{margin:4px 0}}.muted{{color:#666}}</style>
</head><body>
<h1>Automated Report</h1>
<p class="muted">{now}</p>
<h2>Task</h2><pre>{TASK}</pre>
<h2>SQL</h2><pre>{sql}</pre>
<h2>Visualization</h2>
<p><img alt="chart" src="data:image/png;base64,{img_b64}" /></p>
<p class="muted">{caption}</p>
<h2>Data preview</h2>
<pre>{df.head(20).to_markdown(index=False)}</pre>
</body></html>"""
    with open("report.html", "w", encoding="utf-8") as f:
        f.write(html)
    print("Wrote report.html")

if __name__ == "__main__":
    main()
